# Etapa 2 — Visualizaciones interactivas

## Proyecto StreamView Analytics

Este notebook contiene las ocho visualizaciones solicitadas para el análisis del catálogo. Se utiliza Plotly para crear gráficos interactivos y una paleta visual consistente.

In [1]:
import pandas as pd
import plotly.express as px


movies = pd.read_csv("../data/processed/netflix_movies_cleaned.csv")
tv = pd.read_csv("../data/processed/netflix_tv_cleaned.csv")

movies["catalogo"] = "Películas"
tv["catalogo"] = "Series"
catalogo = pd.concat([movies, tv], ignore_index=True)

print(f"Películas: {len(movies):,}")
print(f"Series: {len(tv):,}")
print(f"Catálogo total: {len(catalogo):,}")


Películas: 16,000
Series: 15,991
Catálogo total: 31,991


In [2]:
# Calcular la distribución (mantenemos esta parte igual)
distribucion = catalogo['catalogo'].value_counts().reset_index()
distribucion.columns = ['Tipo', 'Cantidad']

# Crear el gráfico de barras con Plotly
fig_barras = px.bar(
    distribucion, 
    x='Tipo', 
    y='Cantidad', 
    title='Distribución de Películas vs Series',
    color='Tipo',
    color_discrete_map={'Películas': '#E50914', 'Series': "#3c08f7"}, # Colores estilo Netflix
    text_auto=True # Esto coloca el número exacto sobre cada barra
)

# Mejorar el formato visual
fig_barras.update_layout(
    title_font_size=20,
    title_x=0.5, # Centrar el título
    xaxis_title="Tipo de Contenido",
    yaxis_title="Cantidad de Títulos",
    showlegend=False # Ocultamos la leyenda porque el eje X ya dice qué es cada cosa
)

fig_barras.show()

Se eligió el gráfico de barras para comunicar rápidamente la distribución del catálogo, el cual presenta un balance casi equitativo (50/50). Se descartó el uso de gráficos circulares (torta o anillo) ya que la estrecha similitud de los valores dificulta la percepción visual de las proporciones, priorizando así la comparación precisa y directa de las cantidades absolutas.

In [3]:
# 1. Separar, limpiar y contar los géneros
generos_separados = catalogo['genres'].dropna().str.split(',').explode().str.strip()
conteo_generos = generos_separados.value_counts().reset_index()
conteo_generos.columns = ['Género', 'Cantidad']

# 2. Tomar el Top 15 (el Treemap luce mejor con un poco más de datos)
top_generos = conteo_generos.head(15)

# 3. Crear el Treemap con Plotly
fig_treemap = px.treemap(
    top_generos,
    path=[px.Constant("Catálogo"), 'Género'], # Agrega una raíz central para agrupar todo
    values='Cantidad',
    title='Proporción de Géneros Predominantes',
    color='Cantidad', # Colorear según el volumen
    color_continuous_scale='Reds' # Paleta rojiza
)

# 4. Formato visual
fig_treemap.update_layout(
    title_font_size=20,
    title_x=0.5,
    margin=dict(t=50, l=25, r=25, b=25) # Ajustar márgenes para que ocupe bien el espacio
)

# Esto mejora la información al pasar el mouse y dentro de los cuadros
fig_treemap.update_traces(
    textinfo="label+value", 
    hovertemplate='<b>%{label}</b><br>Cantidad: %{value}<extra></extra>'
)

fig_treemap.show()

- Se implementó un Treemap con la finalidad de mostrar de manera óptima la distribución de los géneros, teniendo en cuenta las múltiples categorías que posee el catálogo. Esta visualización maximiza el uso del espacio y permite a la audiencia dimensionar instantáneamente la proporción y el peso de cada nicho de contenido respecto al total.

In [4]:
# ==========================================
# 1. Análisis Geográfico (Top 10 Países)
# ==========================================
# Separar los países, limpiar espacios y contar
paises_separados = catalogo['country'].dropna().str.split(',').explode().str.strip()
top_10_paises = paises_separados.value_counts().head(10).reset_index()
top_10_paises.columns = ['País', 'Cantidad']

# Crear gráfico
fig_paises = px.bar(
    top_10_paises,
    x='País',
    y='Cantidad',
    title='Top 10 Países de Producción',
    text_auto=True,
    color_discrete_sequence=['#1f77b4'] # Azul para el área geográfica
)

fig_paises.update_layout(
    title_font_size=20,
    title_x=0.5,
    xaxis_title="País",
    yaxis_title="Cantidad de Títulos"
)
fig_paises.show()


# ==========================================
# 2. Análisis de Idiomas (Top 10 Idiomas)
# ==========================================
# Separar los idiomas, limpiar espacios y contar
idiomas_separados = catalogo['language'].dropna().str.split(',').explode().str.strip()
top_10_idiomas = idiomas_separados.value_counts().head(10).reset_index()
top_10_idiomas.columns = ['Idioma', 'Cantidad']

# Crear gráfico
fig_idiomas = px.bar(
    top_10_idiomas,
    x='Idioma',
    y='Cantidad',
    title='Top 10 Idiomas de Producción',
    text_auto=True,
    color_discrete_sequence=['#2ca02c'] # Verde para los idiomas
)

fig_idiomas.update_layout(
    title_font_size=20,
    title_x=0.5,
    xaxis_title="Idioma",
    yaxis_title="Cantidad de Títulos"
)
fig_idiomas.show()

- Se optó por utilizar gráficos de barras ordenados de mayor a menor, ya que constituyen la estructura más precisa para comparar las frecuencias absolutas de los datos geográficos y de idioma. Esta jerarquía visual optimiza la carga cognitiva, permitiendo a la audiencia contrastar de manera directa e intuitiva los mercados e idiomas predominantes frente al resto de los valores.

In [5]:
# ==========================================
# 1. Comparativa de popularidad (Boxplot)
# ==========================================
fig_box = px.box(
    catalogo,
    x='catalogo', 
    y='popularity',
    color='catalogo',
    title='Comparativa de Popularidad: Películas vs Series',
    color_discrete_map={'Películas': '#E50914', 'Series': '#564d4d'}
)

# Aplicamos escala logarítmica al eje Y porque la popularidad tiene valores muy extremos
fig_box.update_layout(
    title_font_size=20,
    title_x=0.5,
    yaxis_type="log", 
    xaxis_title="Tipo de Contenido",
    yaxis_title="Popularidad (Escala Logarítmica)",
    showlegend=False
)

fig_box.show()


# ==========================================
# 2. Relación Popularidad vs Calificación (Scatter)
# ==========================================
# Filtramos los que tienen calificación 0 si asumimos que son errores o no tienen votos aún
datos_filtrados = catalogo[catalogo['vote_average'] > 0]

fig_scatter = px.scatter(
    datos_filtrados,
    x='vote_average',
    y='popularity',
    color='catalogo',
    title='Relación entre Popularidad y Calificación',
    color_discrete_map={'Películas': '#E50914', 'Series': '#564d4d'},
    opacity=0.5, # Hacemos los puntos semitransparentes para ver dónde se acumulan
    hover_name='title' # Al pasar el mouse, verás el nombre de la película/serie
)

# También usamos escala logarítmica para Y
fig_scatter.update_layout(
    title_font_size=20,
    title_x=0.5,
    yaxis_type="log",
    xaxis_title="Calificación Promedio (Vote Average)",
    yaxis_title="Popularidad (Escala Logarítmica)"
)

fig_scatter.show()

- Se eligió el boxplot porque permite comparar la distribución de popularidad entre películas y series. Facilita identificar visualmente cuál formato posee una mayor popularidad general (mediana) y detectar rápidamente los valores atípicos, es decir, aquellos títulos o éxitos que sobresalieron extraordinariamente respecto al resto.

- Utilizamos un gráfico de dispersión dado que es la visualización óptima para analizar la relación y correlación entre dos variables continuas (popularidad y calificación promedio). Esto nos permite identificar patrones de comportamiento en la audiencia y responder si los títulos mejor calificados son realmente los más populares."

In [6]:
# Agrupar los datos por año de lanzamiento y tipo de contenido
evolucion_historica = catalogo.groupby(['release_year', 'catalogo']).size().reset_index(name='Cantidad')



# Crear el gráfico de líneas con Plotly
fig_temporal = px.line(
    evolucion_historica,
    x='release_year',
    y='Cantidad',
    color='catalogo',
    title='Evolución Histórica de Lanzamientos (1980 - Presente)',
    color_discrete_map={'Películas': '#E50914', 'Series': '#564d4d'},
    markers=True # Agrega un punto visible en cada año, haciendo más fácil la lectura
)

# Formato visual
fig_temporal.update_layout(
    title_font_size=20,
    title_x=0.5,
    xaxis_title="Año de Lanzamiento",
    yaxis_title="Cantidad de Producciones",
    hovermode="x unified" # Al pasar el mouse, muestra una línea vertical comparando Película y Serie en ese mismo año
)

fig_temporal.show()

- Se seleccionó un gráfico de líneas por ser el estándar óptimo para analizar series temporales y evidenciar tendencias de crecimiento continuo. La aplicación de la paleta corporativa contrastante, junto con los marcadores anuales y el cursor unificado, garantizan una lectura temporal limpia, permitiendo a la gerencia identificar rápidamente las etapas de expansión y los cambios en el volumen de producción histórica.

In [8]:
# ==========================================
# Gráfico A: Presupuesto vs Ingresos (Tamaño Reducido)
# ==========================================
df_finanzas = movies[movies['tiene_datos_financieros']].copy()

fig_finanzas = px.scatter(
    df_finanzas,
    x='budget',
    y='revenue',
    title='Presupuesto vs Ingresos (Punto de Equilibrio)',
    hover_name='title',
    color_discrete_sequence=['#2ca02c'],
    opacity=0.6
)

# Encontrar el valor máximo para trazar la línea de equilibrio
max_val = max(df_finanzas['budget'].max(), df_finanzas['revenue'].max())

# Trazar la línea de equilibrio (y = x)
fig_finanzas.add_shape(
    type="line",
    x0=0, y0=0, x1=max_val, y1=max_val,
    line=dict(color="red", width=2, dash="dash"),
    name="Punto de Equilibrio"
)

# AQUI ESTÁ EL CAMBIO: Se agregaron width y height
fig_finanzas.update_layout(
    title_font_size=20, 
    title_x=0.5,
    xaxis_title="Presupuesto ($)",
    yaxis_title="Ingresos ($)",
    width=700,   # <-- Ancho en píxeles (ajústalo a tu gusto, por defecto suele ser >900)
    height=500   # <-- Alto en píxeles
)

fig_finanzas.show()

# ==========================================
# Gráfico B: Éxito Comercial según ROI (Gráfico de Anillo)
# ==========================================
# 1. Crear una función para clasificar el ROI en palabras simples
def clasificar_roi(roi):
    if roi < 0:
        return "1. Pérdida (No recuperó inversión)"
    elif roi <= 1:
        return "2. Recuperación (Ganancia baja)"
    elif roi <= 3:
        return "3. Éxito Moderado (Ganó hasta el triple)"
    else:
        return "4. Gran Éxito (Multiplicó x3 o más)"

# 2. Aplicar la clasificación a los datos
df_finanzas['Categoria_ROI'] = df_finanzas['ROI'].apply(clasificar_roi)

# 3. Contar cuántas películas hay en cada categoría
conteo_roi = df_finanzas['Categoria_ROI'].value_counts().reset_index()
conteo_roi.columns = ['Categoría', 'Cantidad']
# Ordenar para que aparezcan en un orden lógico
conteo_roi = conteo_roi.sort_values('Categoría')

# 4. Crear el gráfico de anillo con colores semánticos
fig_roi_anillo = px.pie(
    conteo_roi,
    names='Categoría',
    values='Cantidad',
    title='Proporción de Éxito Financiero de las Películas',
    hole=0.4, # Lo convierte en anillo
    color='Categoría',
    color_discrete_map={
        "1. Pérdida (No recuperó inversión)": "#d62728",  # Rojo para pérdidas
        "2. Recuperación (Ganancia baja)": "#ff7f0e",     # Naranja
        "3. Éxito Moderado (Ganó hasta el triple)": "#2ca02c", # Verde claro
        "4. Gran Éxito (Multiplicó x3 o más)": "#1f77b4"  # Azul para los mayores éxitos
    }
)

# Mejorar el formato visual
fig_roi_anillo.update_traces(
    textposition='inside', 
    textinfo='percent+label', # Muestra el porcentaje y el nombre adentro del gráfico
    insidetextorientation='radial'
)

fig_roi_anillo.update_layout(
    title_font_size=20, 
    title_x=0.5,
    showlegend=False, # Ocultamos la leyenda porque las etiquetas ya están en el gráfico
    width=700,  # Hacemos el gráfico un poco más compacto
    height=600
)

fig_roi_anillo.show()

- Se seleccionó un diagrama de dispersión para evaluar la correlación entre la inversión (presupuesto) y el retorno bruto (ingresos). La inclusión de una línea de equilibrio roja discontinua igual al ROI=0  y el uso de opacidad en los datos optimizan la carga visual, permitiendo a la gerencia identificar al instante qué proyectos superaron el umbral de rentabilidad frente a los que generaron pérdidas.

- Se empleó un gráfico de anillo para representar la composición del éxito comercial del catálogo. Al transformar una variable continua compleja (ROI) en categorías discretas de negocio y aplicar una paleta de colores semántica estricta (rojo para pérdidas, verde/azul para ganancias), se facilita la interpretación ejecutiva del riesgo y la rentabilidad en un solo vistazo.